# DeepFruit — Training Notebook

**Sebelum mulai:** Pastikan GPU aktif.
> Runtime → Change runtime type → T4 GPU → Save

Jalankan setiap sel dari atas ke bawah secara berurutan.

## Sel 1 — Cek GPU

In [ ]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPU aktif: {gpus[0].name}")
else:
    print("PERINGATAN: GPU tidak terdeteksi. Aktifkan T4 GPU di Runtime settings!")

## Sel 2 — Setup Kaggle & Download Dataset

Cara dapat API Token:
1. Buka https://www.kaggle.com → klik foto profil → **Settings**
2. Scroll ke bagian **API** → klik **Create New Token**
3. Salin token yang muncul (format: `KGAT_xxxx...`)
4. Paste ke variabel `KAGGLE_TOKEN` di sel berikut, lalu jalankan

In [ ]:
import os

# Paste token kamu di sini (ganti teks di dalam tanda kutip)
KAGGLE_TOKEN = 'KGAT_71cafe1d25d891b7f2ea359892afe8ca'

if KAGGLE_TOKEN == 'PASTE_TOKEN_KAMU_DI_SINI':
    raise ValueError("Ganti PASTE_TOKEN_KAMU_DI_SINI dengan token Kaggle kamu!")

# Set environment variable yang dikenali kaggle CLI versi baru
os.environ['KAGGLE_API_TOKEN'] = KAGGLE_TOKEN
print("Token Kaggle berhasil di-set.")

In [ ]:
# Install kaggle CLI
!pip install -q kaggle

# Download dataset (~1.1 GB, tunggu beberapa menit)
print("Mengunduh dataset... (±1.1 GB, harap tunggu)")
!kaggle datasets download -d sriramr/fruits-fresh-and-rotten-for-classification -p /content/ --unzip
print("Download selesai!")

In [ ]:
# Cek struktur folder dataset
import os

# Cari folder train/test
for root, dirs, files in os.walk('/content'):
    depth = root.count(os.sep) - '/content'.count(os.sep)
    if depth > 3:
        continue
    indent = '  ' * depth
    print(f'{indent}{os.path.basename(root)}/')
    if depth == 3:
        print(f'{indent}  ({len(files)} gambar)')

## Sel 3 — Tentukan Path Dataset

Setelah melihat struktur folder di atas, sesuaikan path `TRAIN_DIR` dan `TEST_DIR` di sel berikut.

In [ ]:
import os

# Cari otomatis folder train dan test
def find_dir(base, name):
    for root, dirs, _ in os.walk(base):
        for d in dirs:
            if d.lower() == name.lower():
                return os.path.join(root, d)
    return None

TRAIN_DIR = find_dir('/content', 'train')
TEST_DIR  = find_dir('/content', 'test')

print(f"TRAIN_DIR: {TRAIN_DIR}")
print(f"TEST_DIR : {TEST_DIR}")

if not TRAIN_DIR or not TEST_DIR:
    print("\nTidak ditemukan otomatis. Isi manual di bawah:")
    # TRAIN_DIR = '/content/xxx/train'
    # TEST_DIR  = '/content/xxx/test'
else:
    classes = sorted(os.listdir(TRAIN_DIR))
    print(f"\n{len(classes)} kelas ditemukan: {classes}")

## Sel 4 — Training Model

In [ ]:
import json
import os
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

OUTPUT_DIR     = '/content/model'
IMG_SIZE       = (224, 224)
BATCH_SIZE     = 32
INITIAL_EPOCHS = 10
FINE_TUNE_EPOCHS = 5
SEED = 123

os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Load data ---
train_ds = keras.utils.image_dataset_from_directory(
    TRAIN_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode='categorical', shuffle=True, seed=SEED,
)
val_ds = keras.utils.image_dataset_from_directory(
    TEST_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode='categorical', shuffle=False,
)

class_names = train_ds.class_names
num_classes = len(class_names)
print(f"\n{num_classes} kelas: {class_names}")

# Simpan class_names sekarang (bisa dipakai meski training gagal di tengah)
with open(os.path.join(OUTPUT_DIR, 'class_names.json'), 'w') as f:
    json.dump(class_names, f, indent=2)
print("class_names.json tersimpan.")

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds   = val_ds.prefetch(AUTOTUNE)

# --- Bangun model ---
data_aug = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
], name='data_augmentation')

base = keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet'
)
base.trainable = False

inputs  = keras.Input(shape=IMG_SIZE + (3,))
x = data_aug(inputs)
x = layers.Rescaling(1./127.5, offset=-1)(x)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)
model   = keras.Model(inputs, outputs)

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

# --- Phase 1: latih head saja ---
print("\n=== Phase 1: melatih classification head ===")
model.fit(train_ds, validation_data=val_ds, epochs=INITIAL_EPOCHS)

# --- Phase 2: fine-tune top layers ---
print("\n=== Phase 2: fine-tuning top 30 layers ===")
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)
model.fit(
    train_ds, validation_data=val_ds,
    epochs=INITIAL_EPOCHS + FINE_TUNE_EPOCHS,
    initial_epoch=INITIAL_EPOCHS,
)

# --- Simpan model ---
model_path = os.path.join(OUTPUT_DIR, 'fruit_model.keras')
model.save(model_path)
print(f"\nModel tersimpan ke: {model_path}")

## Sel 5 — Evaluasi Cepat

In [ ]:
loss, acc = model.evaluate(val_ds, verbose=1)
print(f"\nAkurasi validasi: {acc*100:.2f}%")
print(f"Loss validasi   : {loss:.4f}")

if acc < 0.80:
    print("\nHint: akurasi <80%. Pertimbangkan tambah epoch atau cek kualitas data.")
else:
    print("\nModel siap dipakai!")

## Sel 6 — Download Hasil ke PC

Setelah download, taruh kedua file di:
```
deepfruit/
  model/
    fruit_model.keras   <-- ini
    class_names.json    <-- dan ini
```

In [ ]:
from google.colab import files
import os

model_path = '/content/model/fruit_model.keras'
labels_path = '/content/model/class_names.json'

print("Mengunduh class_names.json...")
files.download(labels_path)

print("Mengunduh fruit_model.keras (bisa beberapa menit, ukuran ~14 MB)...")
files.download(model_path)

print("\nSelesai! Simpan kedua file ke folder model/ di project DeepFruit kamu.")